# FedRod Server

> The core abstraction for `FedRod` server: [https://openreview.net/forum?id=I1hQbx10Kxn](https://openreview.net/forum?id=I1hQbx10Kxn)

In [ ]:
#| default_exp servers.fedrod

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os

import copy
from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.client_selector import BaseClientSelector
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry
from fedai.core import get_clean_kwargs


<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_server("fedrod")
class ServerFedRod(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 
                 

In [ ]:
#| export
@patch
def client_fn(self: ServerFedRod, id, comm_round, client_state):

    if (comm_round == 1 and client_state == {}) or client_state == {}:
        client_state['model'] = copy.deepcopy(self.model.state_dict())
        client_state['head'] = copy.deepcopy(self.model.head.state_dict())

    model = create_model(self.cfg)
    model.load_state_dict(client_state['model'])
    client_state['model'] = model

    head = create_model(self.cfg).head
    head.load_state_dict(client_state['head'])
    client_state['head'] = head
    
    kwargs = get_clean_kwargs(self.cfg.optimizer)
    kwargs.pop("cls", None)
    kwargs.pop("lr", None)
    
    optimizer = get_optimizer(self.cfg)(client_state['model'].parameters(), lr= self.cfg.algorithm.model_lr, **kwargs)
    optimizer.load_state_dict(client_state['optimizer']) if 'optimizer' in client_state else None
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)
    client_state['optimizer'] = optimizer

    opt_head = get_optimizer(self.cfg)(client_state['head'].parameters(), lr= self.cfg.algorithm.head_lr, **kwargs)
    opt_head.load_state_dict(client_state['opt_head']) if 'opt_head' in client_state else None
    for state in opt_head.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)
    client_state['opt_head'] = opt_head

    train_loader = prepare_dl(self.cfg, id, self.fds, train=True, distributed=False)
    test_loader = prepare_dl(self.cfg, id, self.fds, train=False, distributed=False)
    client = self.client_cls(id= id, 
                             cfg= self.cfg,
                             train_loader= train_loader,
                             test_loader= test_loader,
                             state= client_state,
                             criterion= self.criterion,
                             device= self.device,
                             t= comm_round)
    client.logger = self.logger
    return client


### Aggregation


In [ ]:
#| export
@patch
def aggregate(self: ServerFedRod, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    
    with torch.no_grad():
        global_model = None
        
        for id in lst_active_ids:
            client_state_dict = self.state_mgr.get_state(id).get('model', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                global_model[key].add_(client_state_dict[key], alpha=weight)

        self.model.load_state_dict(global_model)
        
        for id in lst_active_ids:
            self.state_mgr.set_state(id, {
                    'model': self.model.state_dict(),
                    'head': self.state_mgr.get_state(id).get('head', None),
                    'optimizer': self.state_mgr.get_state(id).get('optimizer', None),
                    'opt_head': self.state_mgr.get_state(id).get('opt_head', None),
                    'sample_per_class': self.state_mgr.get_state(id).get('sample_per_class', None)
            })
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()